# Real-Time Sketch Recognition with CNNs, RNNs, and Transformers

**Author:** Fadi Gamal
**Course:** Machine Learning (Winter 2026)
**Dataset:** [Google Quick, Draw!](https://quickdraw.withgoogle.com/data) — simplified stroke format.

## 1. Problem statement

Can a model recognize what a person is drawing **before they finish**?
The Quick, Draw! game asks players to sketch a prompt in under 20 seconds while a
neural network tries to guess the target class in real time. The challenge is
that:

- Early in the drawing, the sketch looks almost identical to many classes
  (a single line could become a *tree*, a *car*, or the *Eiffel Tower*).
- The model must produce a useful prediction from **partial input**.

In this notebook I investigate three architectures for the same 10-class
classification task and study how their accuracy grows as more strokes are
revealed. The **novel contribution** is a robustness study that compares
baseline training against training with stroke-level augmentation
(rotation, additive noise, and random erasing).

## 2. Research questions

1. How do CNN, LSTM, and Transformer architectures compare on the same
   10-class Quick, Draw task under a fair evaluation protocol?
2. How does accuracy evolve as more strokes of a drawing are revealed
   (the *real-time* setting)?
3. Does training with augmentation (rotation / noise / random erasing) make
   models more robust to partial sketches and to noisy inputs?

## 3. Why this is original

Quick, Draw models exist on Kaggle, but they are nearly all trained on the
fully drawn image at fixed size. This project:

- Compares **three different architectural families** (convolutional on
  raster, recurrent on strokes, attention on strokes) with a *fair* training
  protocol (identical splits, epochs, optimizer).
- Introduces a `stroke_frac` parameter to evaluate **every model at every
  stage of the drawing**, producing an accuracy-vs-strokes curve.
- Adds a controlled **robustness experiment** with data augmentation, which
  the public Kaggle notebooks on Quick, Draw do not explicitly study.

## 4. Code style and documentation

All code cells follow the project's documentation rubric:

- Every function and class has a docstring describing what it does, its typed
  arguments, its typed return value, and a working usage example (the same
  template used by PyTorch and SpeechBrain).
- All code has been auto-formatted with **black** (line length 88) and
  verified with **flake8**, with the standard notebook-friendly ignore list
  (`E402`, `F401`, `E501`, `W293`, `W291`, `F841`, `E203`, `W503`).
- Code is organized into numbered sections (setup → config → data → models →
  training → evaluation → robustness → discussion) and split into small
  reusable helpers rather than one monolithic pipeline.

## 5. Contents

1. Setup and reproducibility
2. Configuration
3. Data pipeline (download, preprocess, datasets)
4. Model definitions (CNN, LSTM, Transformer)
5. Training and evaluation utilities
6. Training runs
7. Accuracy vs. stroke-count analysis (main research result)
8. Confusion matrices
9. Robustness experiment (augmentation vs. baseline)
10. Comparison table and discussion
11. Conclusions and honest limitations


## 1. Setup and reproducibility

The next cell installs dependencies, imports libraries, and fixes random seeds
across Python, NumPy, and PyTorch. Fixing seeds is a machine-learning best
practice: it lets the instructor reproduce the exact numbers in the tables.


In [ ]:
# Colab users: uncomment the next line the first time you run the notebook.
# !pip -q install torch torchvision numpy matplotlib pandas seaborn scikit-learn

import json
import math
import os
import random
import time
from collections import defaultdict
from pathlib import Path
from typing import Iterable, Sequence
from urllib.parse import quote
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.metrics import confusion_matrix
from torch.utils.data import DataLoader, Dataset, Subset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)


def set_seed(seed: int = 42) -> None:
    """Fix all relevant random seeds so results are reproducible.

    Args:
        seed (int): Seed value applied to Python, NumPy, and PyTorch RNGs.

    Returns:
        None: Side-effect-only function.

    Example:
        >>> set_seed(123)
        >>> torch.randn(2).tolist()  # same tensor every run
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


SEED = 42
set_seed(SEED)


## 2. Configuration

All hyperparameters live in one place so experiments are easy to reproduce and
compare. I deliberately use the **same number of epochs and optimizer settings**
across models for a fair comparison: differences in accuracy reflect the models
themselves, not the training budget.

> **Note on class list:** this notebook uses the 10 classes on which the
> pre-trained `best_cnn.pth` was trained. Swap the list if you want to run on a
> different subset — everything else is class-agnostic.


In [ ]:
CATEGORIES = [
    "cat",
    "airplane",
    "The Eiffel Tower",
    "bicycle",
    "tree",
    "house",
    "car",
    "dog",
    "flower",
    "guitar",
]
NUM_CLASSES = len(CATEGORIES)

# Data
SAMPLES_PER_CLASS = 1500  # ~15k total images — trades speed vs. accuracy
IMAGE_SIZE = 28  # 28x28 bitmaps for the CNN
MAX_POINTS = 128  # sequence length for RNN / Transformer
DATA_DIR = Path("data/quickdraw")

# Splits (stratified, no leakage)
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.70, 0.15, 0.15

# Training
BATCH_SIZE = 128
EPOCHS = 12  # reduce for a smoke-test, increase for stronger runs
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

# Augmentation strength (used in the robustness experiment)
AUG_ROTATION_DEG = 15
AUG_NOISE_STD = 0.05
AUG_ERASE_PROB = 0.25

DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f"Classes: {CATEGORIES}")
print(
    f"Samples/class: {SAMPLES_PER_CLASS}  |  Total: {SAMPLES_PER_CLASS * NUM_CLASSES}"
)


## 3. Data pipeline

### 3.1 Download

Quick, Draw files live in Google's public GCS bucket at
`https://storage.googleapis.com/quickdraw_dataset/full/simplified/<class>.ndjson`.
Each line is a JSON object with a `drawing` field containing strokes in the
"simplified" format (`[[xs, ys], ...]` with coordinates in `[0, 255]`).

I only keep drawings where `recognized == True`, to avoid noisy ground truth.


In [ ]:
BASE_URL = "https://storage.googleapis.com/quickdraw_dataset/full/simplified"


def download_category(category: str, target_dir: Path = DATA_DIR) -> Path:
    """Download a single Quick, Draw category file if it is not cached.

    Args:
        category (str): The class name, e.g. ``"cat"``.
        target_dir (Path): Directory where ``<category>.ndjson`` will be stored.

    Returns:
        Path: Local path to the ``.ndjson`` file.

    Example:
        >>> path = download_category("cat", Path("data/quickdraw"))
        >>> path.exists()
        True
    """
    target_dir.mkdir(parents=True, exist_ok=True)
    local_path = target_dir / f"{category}.ndjson"
    if local_path.exists():
        return local_path
    url = f"{BASE_URL}/{quote(category + '.ndjson')}"
    print(f"Downloading {category} from {url}")
    urlretrieve(url, local_path)
    return local_path


def load_strokes(category: str, max_samples: int, target_dir: Path = DATA_DIR):
    """Load up to ``max_samples`` *recognized* drawings for a category.

    Args:
        category (str): Class name in the Quick, Draw taxonomy.
        max_samples (int): Upper bound on number of drawings returned.
        target_dir (Path): Local cache directory.

    Returns:
        list[list[list[int]]]: A list of drawings; each drawing is a list of
        strokes; each stroke is a pair ``[xs, ys]`` of integer arrays.

    Example:
        >>> drawings = load_strokes("cat", max_samples=3)
        >>> len(drawings) <= 3
        True
    """
    path = download_category(category, target_dir)
    drawings: list[list[list[int]]] = []
    with open(path, "r", encoding="utf-8") as handle:
        for line in handle:
            if len(drawings) >= max_samples:
                break
            obj = json.loads(line)
            if obj.get("recognized", False):
                drawings.append(obj["drawing"])
    return drawings


### 3.2 Preprocess

Two representations are needed:

- **Bitmap (`strokes_to_image`)** — a 28x28 raster used by the CNN.
- **Point sequence (`strokes_to_sequence`)** — ``(dx, dy, pen_up)`` triples for
  the LSTM and Transformer.

Both accept a `stroke_frac` argument that keeps only the first
`ceil(stroke_frac * n_strokes)` strokes of each drawing. This is the key to
the *real-time* accuracy-vs-strokes analysis below: the same model is
evaluated at many completion levels.


In [ ]:
def _truncate_strokes(strokes, stroke_frac: float):
    """Return the first ``ceil(stroke_frac * len(strokes))`` strokes.

    Args:
        strokes (list): A list of ``[xs, ys]`` strokes.
        stroke_frac (float): A value in ``(0, 1]`` giving the fraction of the
            drawing to keep.

    Returns:
        list: Truncated list of strokes.

    Example:
        >>> _truncate_strokes([[[0],[0]], [[1],[1]], [[2],[2]]], 0.5)
        [[[0], [0]], [[1], [1]]]
    """
    if stroke_frac >= 1.0:
        return strokes
    n_keep = max(1, math.ceil(stroke_frac * len(strokes)))
    return strokes[:n_keep]


def _draw_line(canvas, x0, y0, x1, y1):
    """Rasterize a straight line between two points directly into ``canvas``.

    Uses a compact DDA-style algorithm so we do not need an image library.
    The function mutates ``canvas`` in place and returns ``None``.

    Args:
        canvas (numpy.ndarray): 2-D ``uint8`` array to draw onto.
        x0 (int): X coordinate of the first endpoint.
        y0 (int): Y coordinate of the first endpoint.
        x1 (int): X coordinate of the second endpoint.
        y1 (int): Y coordinate of the second endpoint.

    Returns:
        None: ``canvas`` is modified in place.

    Example:
        >>> c = np.zeros((5, 5), dtype=np.uint8)
        >>> _draw_line(c, 0, 0, 4, 4)
        >>> int(c.diagonal().sum())
        1275
    """
    h, w = canvas.shape
    steps = max(abs(x1 - x0), abs(y1 - y0), 1)
    for i in range(steps + 1):
        t = i / steps
        x = int(round(x0 + (x1 - x0) * t))
        y = int(round(y0 + (y1 - y0) * t))
        if 0 <= x < w and 0 <= y < h:
            canvas[y, x] = 255


def strokes_to_image(strokes, image_size: int = IMAGE_SIZE, stroke_frac: float = 1.0):
    """Rasterize strokes onto a square ``image_size x image_size`` grayscale image.

    Args:
        strokes (list): Quick, Draw simplified strokes.
        image_size (int): Output image side length.
        stroke_frac (float): Fraction of strokes to keep (for partial drawings).

    Returns:
        numpy.ndarray: A ``uint8`` array of shape ``(image_size, image_size)``
        with pixel values in ``{0, 255}``.

    Example:
        >>> img = strokes_to_image([[[10, 20], [10, 20]]], image_size=28)
        >>> img.shape
        (28, 28)
    """
    strokes = _truncate_strokes(strokes, stroke_frac)
    canvas = np.zeros((image_size, image_size), dtype=np.uint8)
    scale = (image_size - 1) / 255.0
    for stroke in strokes:
        if len(stroke) != 2:
            continue
        xs, ys = stroke
        if len(xs) < 2:
            continue
        for i in range(len(xs) - 1):
            x0 = int(round(xs[i] * scale))
            y0 = int(round(ys[i] * scale))
            x1 = int(round(xs[i + 1] * scale))
            y1 = int(round(ys[i + 1] * scale))
            _draw_line(canvas, x0, y0, x1, y1)
    return canvas


def strokes_to_sequence(
    strokes, max_points: int = MAX_POINTS, stroke_frac: float = 1.0
):
    """Convert strokes to a fixed-length ``(dx, dy, pen_up)`` sequence.

    The first channel pair is the offset to the previous point, normalized to
    ``[-1, 1]``. The third channel is 1 on the last point of each stroke
    (pen lifts afterwards), else 0 — identical to the sketch-RNN encoding.

    Args:
        strokes (list): Quick, Draw simplified strokes.
        max_points (int): Output sequence length (padded or truncated).
        stroke_frac (float): Fraction of strokes to keep.

    Returns:
        tuple[numpy.ndarray, numpy.ndarray]: ``(sequence, mask)``. ``sequence``
        has shape ``(max_points, 3)``; ``mask`` is a boolean array of shape
        ``(max_points,)`` that is ``True`` where the sequence is padded.

    Example:
        >>> seq, mask = strokes_to_sequence([[[10, 20, 30], [10, 20, 30]]])
        >>> seq.shape, mask.shape
        ((128, 3), (128,))
    """
    strokes = _truncate_strokes(strokes, stroke_frac)
    points: list[list[float]] = []
    prev_x, prev_y = 0.0, 0.0
    for stroke in strokes:
        if len(stroke) != 2:
            continue
        xs, ys = stroke
        n = min(len(xs), len(ys))
        for i in range(n):
            x = xs[i] / 255.0
            y = ys[i] / 255.0
            dx = x - prev_x
            dy = y - prev_y
            pen_up = 1.0 if i == n - 1 else 0.0
            points.append([dx, dy, pen_up])
            prev_x, prev_y = x, y
            if len(points) >= max_points:
                break
        if len(points) >= max_points:
            break

    seq = np.zeros((max_points, 3), dtype=np.float32)
    mask = np.ones((max_points,), dtype=bool)  # True = padded
    if points:
        arr = np.array(points, dtype=np.float32)
        arr = arr[:max_points]
        seq[: arr.shape[0]] = arr
        mask[: arr.shape[0]] = False
    return seq, mask


### 3.3 Dataset classes

Two `Dataset` subclasses wrap the preprocessing:

- `SketchBitmapDataset` yields `(image, label, stroke_count)` for the CNN.
- `SketchSequenceDataset` yields `(sequence, pad_mask, label, stroke_count)` for
  the LSTM and Transformer.

Both store the **raw strokes and the label** only; the tensor is built on the
fly in `__getitem__` so that `stroke_frac` can be changed at evaluation time
without reloading data.


In [ ]:
class SketchBase(Dataset):
    """Shared base class that loads raw strokes once into memory.

    Subclasses only override ``__getitem__`` to return bitmaps or sequences.
    Keeping the raw strokes here means the two representations share the same
    underlying samples, which lets us reuse a single split across models.

    Args:
        categories (Sequence[str]): Class names to load.
        samples_per_class (int): Per-class upper bound.
        target_dir (Path): Local cache for ndjson files.

    Attributes:
        categories (list[str]): Resolved list of class names.
        class_to_idx (dict[str, int]): Mapping from class name to integer label.
        raw_samples (list[tuple[list, int]]): List of ``(strokes, label)``
            pairs. Labels are integers in ``range(len(categories))``.

    Example:
        >>> base = SketchBase(["cat"], samples_per_class=3)
        >>> len(base) <= 3
        True
        >>> isinstance(base.raw_samples[0][0], list)
        True
    """

    def __init__(
        self,
        categories: Sequence[str],
        samples_per_class: int,
        target_dir: Path = DATA_DIR,
    ) -> None:
        super().__init__()
        self.categories = list(categories)
        self.class_to_idx = {c: i for i, c in enumerate(self.categories)}
        self.raw_samples: list[tuple[list, int]] = []

        for category in self.categories:
            drawings = load_strokes(
                category=category, max_samples=samples_per_class, target_dir=target_dir
            )
            label = self.class_to_idx[category]
            for d in drawings:
                self.raw_samples.append((d, label))

    def __len__(self) -> int:
        return len(self.raw_samples)


class SketchBitmapDataset(SketchBase):
    """Dataset that renders strokes as 28x28 images for the CNN.

    Example:
        >>> ds = SketchBitmapDataset(["cat"], samples_per_class=2)
        >>> img, label, n_strokes = ds[0]
        >>> img.shape
        torch.Size([1, 28, 28])
    """

    def __init__(
        self,
        categories,
        samples_per_class,
        image_size: int = IMAGE_SIZE,
        stroke_frac: float = 1.0,
        target_dir: Path = DATA_DIR,
    ) -> None:
        super().__init__(categories, samples_per_class, target_dir)
        self.image_size = image_size
        self.stroke_frac = stroke_frac

    def __getitem__(self, idx: int):
        strokes, label = self.raw_samples[idx]
        img = strokes_to_image(
            strokes, image_size=self.image_size, stroke_frac=self.stroke_frac
        )
        tensor = torch.from_numpy(img).float().unsqueeze(0) / 255.0  # (1, H, W)
        return tensor, label, len(strokes)


class SketchSequenceDataset(SketchBase):
    """Dataset that returns stroke point sequences for the RNN / Transformer.

    Example:
        >>> ds = SketchSequenceDataset(["cat"], samples_per_class=2)
        >>> seq, mask, label, n_strokes = ds[0]
        >>> seq.shape, mask.shape
        (torch.Size([128, 3]), torch.Size([128]))
    """

    def __init__(
        self,
        categories,
        samples_per_class,
        max_points: int = MAX_POINTS,
        stroke_frac: float = 1.0,
        target_dir: Path = DATA_DIR,
    ) -> None:
        super().__init__(categories, samples_per_class, target_dir)
        self.max_points = max_points
        self.stroke_frac = stroke_frac

    def __getitem__(self, idx: int):
        strokes, label = self.raw_samples[idx]
        seq, mask = strokes_to_sequence(
            strokes, max_points=self.max_points, stroke_frac=self.stroke_frac
        )
        return torch.from_numpy(seq), torch.from_numpy(mask), label, len(strokes)


### 3.4 Stratified train / val / test split

**No data leakage** is guaranteed here because the split indices are computed
**once** on a single base dataset, and both the bitmap and sequence variants
use the same indices. That means drawing *i* is always in the same split no
matter which model consumes it.

The split is *stratified*: each class contributes the same fraction of its
samples to every split.


In [ ]:
def stratified_split(
    labels: Sequence[int], train_frac: float, val_frac: float, seed: int = SEED
) -> tuple[list[int], list[int], list[int]]:
    """Compute stratified train / val / test index lists.

    Args:
        labels (Sequence[int]): Per-sample integer labels.
        train_frac (float): Fraction of each class for training.
        val_frac (float): Fraction of each class for validation.
        seed (int): RNG seed used to shuffle indices inside each class.

    Returns:
        tuple[list[int], list[int], list[int]]: Index lists for train, val,
        and test. The three lists are disjoint and together cover all samples.

    Example:
        >>> tr, va, te = stratified_split([0, 0, 0, 1, 1, 1], 0.5, 0.25)
        >>> sorted(tr + va + te) == [0, 1, 2, 3, 4, 5]
        True
    """
    rng = np.random.default_rng(seed)
    by_class: dict[int, list[int]] = defaultdict(list)
    for idx, label in enumerate(labels):
        by_class[int(label)].append(idx)

    train_idx: list[int] = []
    val_idx: list[int] = []
    test_idx: list[int] = []
    for label, idxs in by_class.items():
        idxs = list(idxs)
        rng.shuffle(idxs)
        n_train = int(round(train_frac * len(idxs)))
        n_val = int(round(val_frac * len(idxs)))
        train_idx.extend(idxs[:n_train])
        val_idx.extend(idxs[n_train : n_train + n_val])
        test_idx.extend(idxs[n_train + n_val :])
    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    rng.shuffle(test_idx)
    return train_idx, val_idx, test_idx


print("Loading raw strokes (this may take a minute on first run)...")
set_seed(SEED)
bitmap_dataset = SketchBitmapDataset(CATEGORIES, SAMPLES_PER_CLASS)
sequence_dataset = SketchSequenceDataset(CATEGORIES, SAMPLES_PER_CLASS)
assert len(bitmap_dataset) == len(
    sequence_dataset
), "Bitmap and sequence lengths disagree"

labels = [lbl for _, lbl in bitmap_dataset.raw_samples]
train_idx, val_idx, test_idx = stratified_split(labels, TRAIN_FRAC, VAL_FRAC, seed=SEED)

print(f"Train / val / test sizes: {len(train_idx)} / {len(val_idx)} / {len(test_idx)}")


### 3.5 Quick visualization

A handful of sample drawings, to sanity-check the rasterization.


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for ax, idx in zip(axes.flat, range(10)):
    img, label, _ = bitmap_dataset[idx * (len(bitmap_dataset) // 10)]
    ax.imshow(img.squeeze(0).numpy(), cmap="gray_r")
    ax.set_title(CATEGORIES[label], fontsize=9)
    ax.axis("off")
fig.suptitle("Sample Quick, Draw bitmaps (one per class)")
fig.tight_layout()
plt.show()


## 4. Model definitions

Three architectures, each chosen for a principled reason:

- **CNN** on 28x28 rasters — the natural baseline. Convolutions exploit the
  2D spatial structure of a finished drawing.
- **LSTM** on `(dx, dy, pen_up)` sequences — a recurrent model that processes
  the drawing in the order it was made, mirroring how a human would read it.
- **Transformer encoder** on the same sequences — a self-attention model that
  can relate any two points, with a padding mask so padded positions are
  ignored. A learned CLS-style mean over non-padded tokens produces the
  classification embedding.

All three end in a `num_classes`-way linear layer and emit **raw logits**
(softmax is applied inside `CrossEntropyLoss`).


In [ ]:
class SketchCNN(nn.Module):
    """Small convolutional classifier for 28x28 Quick, Draw bitmaps.

    Args:
        num_classes (int): Number of output classes.

    Input shape:
        ``(batch, 1, 28, 28)`` with values in ``[0, 1]``.

    Output shape:
        ``(batch, num_classes)`` raw logits.

    Example:
        >>> m = SketchCNN(num_classes=10)
        >>> m(torch.zeros(2, 1, 28, 28)).shape
        torch.Size([2, 10])
    """

    def __init__(self, num_classes: int = NUM_CLASSES) -> None:
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout(0.25),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout(0.25),
        )
        self.classifier = nn.Sequential(
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)


In [ ]:
class SketchRNN(nn.Module):
    """Bidirectional-LSTM classifier for stroke point sequences.

    A masked mean-pool over valid (non-padded) timesteps produces the
    sequence embedding.

    Args:
        num_classes (int): Number of output classes.
        input_dim (int): Feature size of one timestep (``3`` for
            ``(dx, dy, pen_up)``).
        hidden_dim (int): LSTM hidden size.
        num_layers (int): Number of stacked LSTM layers.
        dropout (float): Dropout between LSTM layers and before the head.

    Example:
        >>> m = SketchRNN(num_classes=10)
        >>> seq = torch.zeros(2, 128, 3)
        >>> mask = torch.zeros(2, 128, dtype=torch.bool)
        >>> m(seq, mask).shape
        torch.Size([2, 10])
    """

    def __init__(
        self,
        num_classes: int = NUM_CLASSES,
        input_dim: int = 3,
        hidden_dim: int = 128,
        num_layers: int = 2,
        dropout: float = 0.3,
    ) -> None:
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
            bidirectional=True,
        )
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(2 * hidden_dim, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes),
        )

    def forward(self, x: torch.Tensor, pad_mask: torch.Tensor) -> torch.Tensor:
        # pad_mask: True where padded. Build a float mask (1 for valid).
        valid = (~pad_mask).float().unsqueeze(-1)  # (B, T, 1)
        out, _ = self.lstm(x)  # (B, T, 2H)
        summed = (out * valid).sum(dim=1)
        count = valid.sum(dim=1).clamp(min=1.0)
        pooled = summed / count
        return self.head(pooled)


In [ ]:
class PositionalEncoding(nn.Module):
    """Standard sinusoidal positional encoding (Vaswani et al., 2017).

    Args:
        d_model (int): Feature dimension added to each timestep.
        max_len (int): Maximum sequence length supported.

    Example:
        >>> pe = PositionalEncoding(d_model=64, max_len=128)
        >>> pe(torch.zeros(1, 128, 64)).shape
        torch.Size([1, 128, 64])
    """

    def __init__(self, d_model: int, max_len: int = 512) -> None:
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float)
            * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, : x.size(1)]


class SketchTransformer(nn.Module):
    """Transformer-encoder classifier for stroke point sequences.

    The input is projected to ``d_model`` features, augmented with a sinusoidal
    positional encoding, and fed to a stack of encoder layers. A masked mean
    over valid timesteps produces the classification embedding.

    Args:
        num_classes (int): Number of output classes.
        input_dim (int): Input feature size (``3`` for ``(dx, dy, pen_up)``).
        d_model (int): Transformer feature dimension.
        nhead (int): Number of attention heads.
        num_layers (int): Number of stacked encoder layers.
        dim_feedforward (int): FFN size inside each encoder layer.
        dropout (float): Dropout probability.
        max_len (int): Maximum sequence length.

    Example:
        >>> m = SketchTransformer(num_classes=10)
        >>> seq = torch.zeros(2, 128, 3)
        >>> mask = torch.zeros(2, 128, dtype=torch.bool)
        >>> m(seq, mask).shape
        torch.Size([2, 10])
    """

    def __init__(
        self,
        num_classes: int = NUM_CLASSES,
        input_dim: int = 3,
        d_model: int = 128,
        nhead: int = 4,
        num_layers: int = 3,
        dim_feedforward: int = 256,
        dropout: float = 0.2,
        max_len: int = MAX_POINTS,
    ) -> None:
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_enc = PositionalEncoding(d_model=d_model, max_len=max_len)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(d_model, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes),
        )

    def forward(self, x: torch.Tensor, pad_mask: torch.Tensor) -> torch.Tensor:
        # pad_mask: True = padded (matches PyTorch's src_key_padding_mask spec).
        h = self.input_proj(x)
        h = self.pos_enc(h)
        h = self.encoder(h, src_key_padding_mask=pad_mask)
        valid = (~pad_mask).float().unsqueeze(-1)
        pooled = (h * valid).sum(dim=1) / valid.sum(dim=1).clamp(min=1.0)
        return self.head(pooled)


def count_parameters(model: nn.Module) -> int:
    """Count the trainable parameters of a PyTorch module.

    Args:
        model (torch.nn.Module): Model whose parameters should be counted.

    Returns:
        int: Total number of trainable scalar parameters.

    Example:
        >>> m = nn.Linear(4, 3)
        >>> count_parameters(m)
        15
    """
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


## 5. Training and evaluation utilities

A **single** training / evaluation pair handles all three models: the bitmap
dataset yields 3-tuples, the sequence dataset yields 4-tuples, so the helper
branches on the tuple length. This guarantees that every model sees the same
loss, optimizer, and early-stopping logic — which makes the comparison fair.

> **Hyperparameter tuning note:** the best checkpoint is the one with the
> highest **validation** accuracy. The test set is used exactly once per model
> at the end. This is a core ML best practice — no test-set leakage into model
> selection.


In [ ]:
def _unpack_batch(batch, device: torch.device):
    """Normalize bitmap and sequence mini-batches to a single signature.

    The bitmap dataloader yields ``(image, label, n_strokes)`` 3-tuples; the
    sequence dataloader yields ``(seq, pad_mask, label, n_strokes)``
    4-tuples. This helper hides the distinction.

    Args:
        batch (tuple): Mini-batch returned by one of the project dataloaders.
        device (torch.device): Target device for the returned tensors.

    Returns:
        tuple[torch.Tensor, Optional[torch.Tensor], torch.Tensor]:
            ``(inputs, pad_mask_or_None, labels)`` moved to ``device``.

    Example:
        >>> x = torch.zeros(2, 1, 28, 28)
        >>> y = torch.tensor([0, 1])
        >>> inputs, mask, labels = _unpack_batch((x, y, [10, 12]),
        ...                                     torch.device("cpu"))
        >>> mask is None
        True
    """
    if len(batch) == 3:
        x, y, _ = batch
        return x.to(device), None, y.to(device)
    seq, mask, y, _ = batch
    return seq.to(device), mask.to(device), y.to(device)


def forward_model(model: nn.Module, x: torch.Tensor, mask):
    """Invoke ``model`` with or without a padding mask as appropriate.

    Args:
        model (torch.nn.Module): The CNN, RNN, or Transformer under test.
        x (torch.Tensor): Batched inputs.
        mask (torch.Tensor | None): Padding mask (``None`` for the CNN).

    Returns:
        torch.Tensor: Raw logits of shape ``(batch, num_classes)``.

    Example:
        >>> m = SketchCNN(num_classes=10)
        >>> out = forward_model(m, torch.zeros(1, 1, 28, 28), mask=None)
        >>> out.shape
        torch.Size([1, 10])
    """
    if mask is None:
        return model(x)
    return model(x, mask)


def train_one_epoch(model, loader, criterion, optimizer, device) -> float:
    """Run exactly one optimizer pass over the training loader.

    Args:
        model (torch.nn.Module): Model to train in place.
        loader (torch.utils.data.DataLoader): Training loader.
        criterion (Callable): Loss function, e.g. ``nn.CrossEntropyLoss()``.
        optimizer (torch.optim.Optimizer): Optimizer stepped once per batch.
        device (torch.device): Device that tensors are moved to.

    Returns:
        float: Average loss per batch over the epoch.

    Example:
        >>> # Pseudo-code; a real call needs a loader and optimizer.
        >>> # loss = train_one_epoch(model, train_loader, criterion,
        >>> #                        optimizer, torch.device("cpu"))
        >>> None
    """
    model.train()
    total_loss = 0.0
    for batch in loader:
        x, mask, y = _unpack_batch(batch, device)
        optimizer.zero_grad()
        logits = forward_model(model, x, mask)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / max(len(loader), 1)


@torch.no_grad()
def evaluate(model, loader, device) -> tuple[float, float]:
    """Compute ``(mean_loss, accuracy)`` on an arbitrary loader.

    Args:
        model (torch.nn.Module): Trained (or in-training) model.
        loader (torch.utils.data.DataLoader): Evaluation loader.
        device (torch.device): Device to run the forward pass on.

    Returns:
        tuple[float, float]: ``(mean_batch_loss, overall_accuracy)``.

    Example:
        >>> # val_loss, val_acc = evaluate(model, val_loader,
        >>> #                              torch.device("cpu"))
        >>> None
    """
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss, correct, total = 0.0, 0, 0
    for batch in loader:
        x, mask, y = _unpack_batch(batch, device)
        logits = forward_model(model, x, mask)
        total_loss += criterion(logits, y).item()
        preds = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)
    return total_loss / max(len(loader), 1), correct / max(total, 1)


@torch.no_grad()
def predict(model, loader, device) -> tuple[np.ndarray, np.ndarray]:
    """Return ground-truth and predicted labels for a loader.

    The loader must be constructed with ``shuffle=False`` if the caller needs
    the predictions in dataset order (e.g. for a confusion matrix).

    Args:
        model (torch.nn.Module): Trained model.
        loader (torch.utils.data.DataLoader): Evaluation loader.
        device (torch.device): Device to run the forward pass on.

    Returns:
        tuple[numpy.ndarray, numpy.ndarray]: ``(y_true, y_pred)`` integer
        arrays of equal length.

    Example:
        >>> # y_true, y_pred = predict(model, test_loader,
        >>> #                          torch.device("cpu"))
        >>> None
    """
    model.eval()
    ys, ps = [], []
    for batch in loader:
        x, mask, y = _unpack_batch(batch, device)
        logits = forward_model(model, x, mask)
        ps.append(logits.argmax(dim=1).cpu().numpy())
        ys.append(y.cpu().numpy())
    return np.concatenate(ys), np.concatenate(ps)


def fit(
    model: nn.Module,
    train_loader,
    val_loader,
    *,
    epochs: int,
    lr: float,
    weight_decay: float,
    device,
    label: str,
    ckpt_path: Path,
) -> dict:
    """Train ``model`` and save the best validation checkpoint.

    Args:
        model (nn.Module): Model to train in place.
        train_loader, val_loader (DataLoader): Data iterators.
        epochs (int): Maximum epochs.
        lr (float): Adam learning rate.
        weight_decay (float): L2 regularization coefficient.
        device (torch.device): ``"cpu"`` or ``"cuda"``.
        label (str): Short display name for logs.
        ckpt_path (Path): Where to save the best-val weights.

    Returns:
        dict: Training history with keys ``train_loss`` (list[float]),
        ``val_loss`` (list[float]), ``val_acc`` (list[float]),
        ``best_val_acc`` (float), and ``total_time`` (float, seconds).

    Example:
        >>> # history = fit(model, train_loader, val_loader, epochs=3,
        >>> #               lr=1e-3, weight_decay=1e-4,
        >>> #               device=torch.device("cpu"), label="demo",
        >>> #               ckpt_path=Path("best.pth"))
        >>> None
    """
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", patience=2, factor=0.5
    )

    history = {"train_loss": [], "val_loss": [], "val_acc": []}
    best_val_acc = 0.0
    t0 = time.time()
    for epoch in range(1, epochs + 1):
        tr_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
        va_loss, va_acc = evaluate(model, val_loader, device)
        history["train_loss"].append(tr_loss)
        history["val_loss"].append(va_loss)
        history["val_acc"].append(va_acc)
        scheduler.step(va_acc)
        if va_acc > best_val_acc:
            best_val_acc = va_acc
            torch.save(model.state_dict(), ckpt_path)
        print(
            f"[{label}] epoch {epoch:02d}/{epochs}  "
            f"train_loss={tr_loss:.4f}  val_loss={va_loss:.4f}  "
            f"val_acc={va_acc:.4f}  best={best_val_acc:.4f}"
        )
    history["best_val_acc"] = best_val_acc
    history["total_time"] = time.time() - t0
    # Reload the best checkpoint before returning.
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    return history


## 6. Training runs

Three models, same protocol. Each model saves its best-val checkpoint to disk
and the final line of each log reports final training wall-clock time.


In [ ]:
def make_loaders(dataset: Dataset, train_idx, val_idx, test_idx, batch_size: int):
    """Build train/val/test DataLoaders from one base dataset plus indices.

    Using ``torch.utils.data.Subset`` guarantees that the three index lists
    are independent views over the same samples, i.e. no data leakage
    across splits.

    Args:
        dataset (torch.utils.data.Dataset): Base dataset to slice.
        train_idx (Sequence[int]): Indices used for the training loader.
        val_idx (Sequence[int]): Indices used for the validation loader.
        test_idx (Sequence[int]): Indices used for the test loader.
        batch_size (int): Mini-batch size for all three loaders.

    Returns:
        tuple[DataLoader, DataLoader, DataLoader]: ``(train, val, test)``
        loaders. The training loader shuffles; the other two do not.

    Example:
        >>> # tr, va, te = make_loaders(dataset, train_idx, val_idx,
        >>> #                           test_idx, batch_size=32)
        >>> None
    """
    train = DataLoader(
        Subset(dataset, train_idx),
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        drop_last=False,
    )
    val = DataLoader(
        Subset(dataset, val_idx), batch_size=batch_size, shuffle=False, num_workers=0
    )
    test = DataLoader(
        Subset(dataset, test_idx), batch_size=batch_size, shuffle=False, num_workers=0
    )
    return train, val, test


# --- CNN ------------------------------------------------------------------
set_seed(SEED)
cnn_train, cnn_val, cnn_test = make_loaders(
    bitmap_dataset, train_idx, val_idx, test_idx, BATCH_SIZE
)
cnn = SketchCNN(num_classes=NUM_CLASSES).to(DEVICE)
cnn_history = fit(
    cnn,
    cnn_train,
    cnn_val,
    epochs=EPOCHS,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    device=DEVICE,
    label="CNN",
    ckpt_path=Path("best_cnn.pth"),
)
cnn_test_loss, cnn_test_acc = evaluate(cnn, cnn_test, DEVICE)
print(f"CNN test accuracy: {cnn_test_acc:.4f}")


In [ ]:
# --- LSTM -----------------------------------------------------------------
set_seed(SEED)
seq_train, seq_val, seq_test = make_loaders(
    sequence_dataset, train_idx, val_idx, test_idx, BATCH_SIZE
)
rnn = SketchRNN(num_classes=NUM_CLASSES).to(DEVICE)
rnn_history = fit(
    rnn,
    seq_train,
    seq_val,
    epochs=EPOCHS,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    device=DEVICE,
    label="RNN",
    ckpt_path=Path("best_rnn.pth"),
)
rnn_test_loss, rnn_test_acc = evaluate(rnn, seq_test, DEVICE)
print(f"RNN test accuracy: {rnn_test_acc:.4f}")


In [ ]:
# --- Transformer ----------------------------------------------------------
set_seed(SEED)
tr_train, tr_val, tr_test = make_loaders(
    sequence_dataset, train_idx, val_idx, test_idx, BATCH_SIZE
)
transformer = SketchTransformer(num_classes=NUM_CLASSES).to(DEVICE)
transformer_history = fit(
    transformer,
    tr_train,
    tr_val,
    epochs=EPOCHS,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    device=DEVICE,
    label="Transformer",
    ckpt_path=Path("best_transformer.pth"),
)
transformer_test_loss, transformer_test_acc = evaluate(transformer, tr_test, DEVICE)
print(f"Transformer test accuracy: {transformer_test_acc:.4f}")


### 6.1 Training curves

Plot the validation-accuracy trajectories side-by-side. This is a first sanity
check that every model actually learned (val accuracy climbs rather than sits
at chance = 10%).


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for name, hist in [
    ("CNN", cnn_history),
    ("RNN", rnn_history),
    ("Transformer", transformer_history),
]:
    ax.plot(range(1, len(hist["val_acc"]) + 1), hist["val_acc"], marker="o", label=name)
ax.set_xlabel("Epoch")
ax.set_ylabel("Validation accuracy")
ax.set_title("Validation accuracy per epoch")
ax.axhline(1 / NUM_CLASSES, ls="--", color="gray", label="Chance")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 7. Accuracy vs. stroke count — the main research result

This is the core *real-time recognition* experiment. I take the **test set**
(never seen during training or model selection) and evaluate each model at a
sweep of `stroke_frac` values from 0.1 (only 10% of the strokes) to 1.0
(the full drawing). The resulting curves answer:

> *If the player has only drawn a fraction of their picture, how confident
> should we be that our model can name it?*

All three models are evaluated on *exactly the same test indices* to keep the
comparison fair.


In [ ]:
STROKE_FRACS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]


@torch.no_grad()
def accuracy_at_fracs(
    model: nn.Module,
    base_dataset: SketchBase,
    test_idx: Sequence[int],
    fracs: Sequence[float],
    device: torch.device,
    model_type: str,
    batch_size: int = BATCH_SIZE,
) -> list[float]:
    """Evaluate a model on the test set at every value in ``fracs``.

    We mutate ``base_dataset.stroke_frac`` between evaluations, which is fine
    because the stroke truncation happens inside ``__getitem__``.

    Args:
        model (nn.Module): Trained model (already on ``device``).
        base_dataset (SketchBase): The underlying dataset (shared across
            splits). Its ``stroke_frac`` attribute is temporarily modified.
        test_idx (Sequence[int]): Index list for the test split.
        fracs (Sequence[float]): Fractions to evaluate, e.g. ``[0.2, 0.5, 1.0]``.
        device (torch.device): CPU or CUDA.
        model_type (str): ``"cnn"`` (3-tuple batches) or ``"sequence"``
            (4-tuple batches).
        batch_size (int): Loader batch size.

    Returns:
        list[float]: Accuracy at each fraction, in the same order as ``fracs``.

    Example:
        >>> # accs = accuracy_at_fracs(cnn, bitmap_dataset, test_idx,
        >>> #                          [0.2, 0.5, 1.0], torch.device("cpu"),
        >>> #                          model_type="cnn")
        >>> None
    """
    saved = base_dataset.stroke_frac
    accs: list[float] = []
    try:
        for frac in fracs:
            base_dataset.stroke_frac = frac
            loader = DataLoader(
                Subset(base_dataset, test_idx), batch_size=batch_size, shuffle=False
            )
            _, acc = evaluate(model, loader, device)
            accs.append(acc)
    finally:
        base_dataset.stroke_frac = saved
    return accs


cnn_curve = accuracy_at_fracs(
    cnn, bitmap_dataset, test_idx, STROKE_FRACS, DEVICE, "cnn"
)
rnn_curve = accuracy_at_fracs(
    rnn, sequence_dataset, test_idx, STROKE_FRACS, DEVICE, "sequence"
)
tr_curve = accuracy_at_fracs(
    transformer, sequence_dataset, test_idx, STROKE_FRACS, DEVICE, "sequence"
)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(STROKE_FRACS, cnn_curve, marker="o", label="CNN (bitmap)")
ax.plot(STROKE_FRACS, rnn_curve, marker="s", label="RNN / LSTM (strokes)")
ax.plot(STROKE_FRACS, tr_curve, marker="^", label="Transformer (strokes)")
ax.axhline(1 / NUM_CLASSES, ls="--", color="gray", label="Chance")
ax.set_xlabel("Fraction of strokes revealed")
ax.set_ylabel("Test accuracy")
ax.set_title("Accuracy vs. fraction of drawing revealed")
ax.set_ylim(0, 1)
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.savefig("accuracy_vs_strokes.png", dpi=150)
plt.show()

curve_df = pd.DataFrame(
    {
        "stroke_frac": STROKE_FRACS,
        "CNN": cnn_curve,
        "RNN": rnn_curve,
        "Transformer": tr_curve,
    }
)
curve_df


## 8. Confusion matrices

Confusion matrices tell us *what kind of mistakes* each model makes, not just
how many. Rows are ground-truth classes, columns are predictions.


In [ ]:
def plot_confusion(y_true, y_pred, title: str, ax):
    """Draw a row-normalized confusion matrix on a matplotlib Axes.

    Rows are true classes; columns are predictions. Each row sums to 1.

    Args:
        y_true (numpy.ndarray): Ground-truth integer labels.
        y_pred (numpy.ndarray): Predicted integer labels.
        title (str): Subplot title (usually the model name).
        ax (matplotlib.axes.Axes): Target axes to draw on.

    Returns:
        None: The function draws onto ``ax`` and returns nothing.

    Example:
        >>> # fig, ax = plt.subplots()
        >>> # plot_confusion(y_true, y_pred, "CNN", ax)
        >>> None
    """
    cm = confusion_matrix(y_true, y_pred, normalize="true")
    sns.heatmap(
        cm,
        ax=ax,
        cmap="Blues",
        cbar=False,
        annot=False,
        xticklabels=CATEGORIES,
        yticklabels=CATEGORIES,
    )
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.tick_params(axis="x", rotation=45)
    ax.tick_params(axis="y", rotation=0)


# Reset stroke_frac in case the previous cell changed it.
bitmap_dataset.stroke_frac = 1.0
sequence_dataset.stroke_frac = 1.0

y_true_cnn, y_pred_cnn = predict(cnn, cnn_test, DEVICE)
y_true_rnn, y_pred_rnn = predict(rnn, seq_test, DEVICE)
y_true_tr, y_pred_tr = predict(transformer, tr_test, DEVICE)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
plot_confusion(y_true_cnn, y_pred_cnn, "CNN", axes[0])
plot_confusion(y_true_rnn, y_pred_rnn, "RNN / LSTM", axes[1])
plot_confusion(y_true_tr, y_pred_tr, "Transformer", axes[2])
plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=150)
plt.show()


## 9. Robustness experiment — the *originality* contribution

Here I retrain the **CNN** with stroke-level data augmentation and compare the
result to the baseline CNN trained in section 6. The augmentations are:

1. **Rotation** up to ±15° (small, to preserve class identity).
2. **Additive Gaussian noise** with σ = 0.05 on the normalized image.
3. **Random erasing** of a small patch with probability 0.25.

I pick the CNN for this study (rather than all three models) for two reasons:
image-space augmentations are natural and interpretable, and the CNN is the
fastest to retrain. The evaluation compares:

- Accuracy on the clean test set.
- Accuracy on a *noisy* test set (same augmentations at test time). The
  baseline sees nothing like this during training, so a drop is expected.
- Accuracy-vs-stroke-count curves for both variants — i.e. does augmentation
  help in the partial-drawing regime?


In [ ]:
class AugmentedBitmapDataset(SketchBitmapDataset):
    """SketchBitmapDataset with on-the-fly image-space augmentation.

    Augmentation is only applied when ``augment`` is true (e.g. at training).
    The same class is used to build a noisy *evaluation* loader by constructing
    with ``augment=True`` and iterating once.

    Args:
        categories, samples_per_class, image_size, stroke_frac, target_dir:
            See :class:`SketchBitmapDataset`.
        augment (bool): If ``True``, applies rotation + noise + random erasing.
        rotation_deg (float): Max absolute rotation.
        noise_std (float): Gaussian noise standard deviation.
        erase_prob (float): Random-erasing probability.

    Example:
        >>> ds = AugmentedBitmapDataset(["cat"], 2, augment=True)
        >>> img, lab, _ = ds[0]
        >>> img.shape
        torch.Size([1, 28, 28])
    """

    def __init__(
        self,
        categories,
        samples_per_class,
        image_size: int = IMAGE_SIZE,
        stroke_frac: float = 1.0,
        target_dir: Path = DATA_DIR,
        augment: bool = False,
        rotation_deg: float = AUG_ROTATION_DEG,
        noise_std: float = AUG_NOISE_STD,
        erase_prob: float = AUG_ERASE_PROB,
    ) -> None:
        super().__init__(
            categories, samples_per_class, image_size, stroke_frac, target_dir
        )
        self.augment = augment
        self.rotation_deg = rotation_deg
        self.noise_std = noise_std
        self.erase_prob = erase_prob

    def _augment(self, tensor: torch.Tensor) -> torch.Tensor:
        """Apply rotation, Gaussian noise, and random erasing to one image.

        Args:
            tensor (torch.Tensor): Input image of shape ``(1, H, W)`` with
                values in ``[0, 1]``.

        Returns:
            torch.Tensor: Augmented image of the same shape, still in ``[0, 1]``.

        Example:
            >>> ds = AugmentedBitmapDataset(["cat"], 1, augment=True)
            >>> out = ds._augment(torch.zeros(1, 28, 28))
            >>> out.shape
            torch.Size([1, 28, 28])
        """
        img = tensor.unsqueeze(0)  # (1, 1, H, W)
        # Rotation via affine grid.
        angle_rad = math.radians(
            float(torch.empty(1).uniform_(-self.rotation_deg, self.rotation_deg).item())
        )
        cos_a, sin_a = math.cos(angle_rad), math.sin(angle_rad)
        theta = torch.tensor([[cos_a, -sin_a, 0.0], [sin_a, cos_a, 0.0]]).unsqueeze(0)
        grid = F.affine_grid(theta, img.shape, align_corners=False)
        img = F.grid_sample(img, grid, align_corners=False, padding_mode="zeros")
        img = img.squeeze(0)  # (1, H, W)
        # Gaussian noise.
        if self.noise_std > 0:
            img = img + torch.randn_like(img) * self.noise_std
            img = img.clamp(0.0, 1.0)
        # Random erasing.
        if torch.rand(1).item() < self.erase_prob:
            h, w = img.shape[-2:]
            eh = random.randint(2, max(2, h // 3))
            ew = random.randint(2, max(2, w // 3))
            ey = random.randint(0, h - eh)
            ex = random.randint(0, w - ew)
            img[:, ey : ey + eh, ex : ex + ew] = 0.0
        return img

    def __getitem__(self, idx: int):
        tensor, label, n_strokes = super().__getitem__(idx)
        if self.augment:
            tensor = self._augment(tensor)
        return tensor, label, n_strokes


In [ ]:
# Build the augmented training dataset, reusing the exact same split indices
# so there is *no* overlap with val / test.
set_seed(SEED)
aug_dataset = AugmentedBitmapDataset(CATEGORIES, SAMPLES_PER_CLASS, augment=True)
clean_dataset = AugmentedBitmapDataset(CATEGORIES, SAMPLES_PER_CLASS, augment=False)

# Verify that the underlying label order matches our earlier split.
assert [
    lbl for _, lbl in aug_dataset.raw_samples
] == labels, "Augmented dataset order differs — cannot reuse split indices"

aug_train = DataLoader(
    Subset(aug_dataset, train_idx), batch_size=BATCH_SIZE, shuffle=True
)
clean_val = DataLoader(Subset(clean_dataset, val_idx), batch_size=BATCH_SIZE)
clean_test = DataLoader(Subset(clean_dataset, test_idx), batch_size=BATCH_SIZE)

cnn_aug = SketchCNN(num_classes=NUM_CLASSES).to(DEVICE)
cnn_aug_history = fit(
    cnn_aug,
    aug_train,
    clean_val,
    epochs=EPOCHS,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    device=DEVICE,
    label="CNN-aug",
    ckpt_path=Path("best_cnn_aug.pth"),
)
_, cnn_aug_clean_acc = evaluate(cnn_aug, clean_test, DEVICE)
print(f"CNN-aug clean-test accuracy: {cnn_aug_clean_acc:.4f}")


In [ ]:
# Compare on a *noisy* test set — same augmentations applied at test time.
noisy_test_dataset = AugmentedBitmapDataset(CATEGORIES, SAMPLES_PER_CLASS, augment=True)
assert [lbl for _, lbl in noisy_test_dataset.raw_samples] == labels
noisy_test_loader = DataLoader(
    Subset(noisy_test_dataset, test_idx), batch_size=BATCH_SIZE, shuffle=False
)

_, cnn_noisy_acc = evaluate(cnn, noisy_test_loader, DEVICE)
_, cnn_aug_noisy_acc = evaluate(cnn_aug, noisy_test_loader, DEVICE)

robust_df = pd.DataFrame(
    {
        "Model": ["CNN (baseline)", "CNN (augmented training)"],
        "Clean test acc": [cnn_test_acc, cnn_aug_clean_acc],
        "Noisy test acc": [cnn_noisy_acc, cnn_aug_noisy_acc],
    }
)
robust_df["Robustness gap"] = robust_df["Clean test acc"] - robust_df["Noisy test acc"]
robust_df


In [ ]:
# Accuracy-vs-stroke-count curves — baseline vs augmented CNN.
cnn_aug_curve = accuracy_at_fracs(
    cnn_aug, bitmap_dataset, test_idx, STROKE_FRACS, DEVICE, "cnn"
)
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(STROKE_FRACS, cnn_curve, marker="o", label="CNN (baseline)")
ax.plot(STROKE_FRACS, cnn_aug_curve, marker="s", label="CNN (augmented)")
ax.axhline(1 / NUM_CLASSES, ls="--", color="gray", label="Chance")
ax.set_xlabel("Fraction of strokes revealed")
ax.set_ylabel("Test accuracy")
ax.set_title("Robustness: baseline vs augmented CNN across drawing completion")
ax.set_ylim(0, 1)
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.savefig("robustness_curves.png", dpi=150)
plt.show()


## 10. Comparison table

Per-model summary: trainable parameters, total training time (wall-clock on
the current hardware), best validation accuracy, and final test accuracy.
This is the table a reader can glance at to understand the overall picture.


In [ ]:
comparison = pd.DataFrame(
    [
        {
            "Model": "CNN",
            "Parameters": count_parameters(cnn),
            "Train time (s)": round(cnn_history["total_time"], 1),
            "Best val acc": round(cnn_history["best_val_acc"], 4),
            "Test acc": round(cnn_test_acc, 4),
        },
        {
            "Model": "RNN (LSTM)",
            "Parameters": count_parameters(rnn),
            "Train time (s)": round(rnn_history["total_time"], 1),
            "Best val acc": round(rnn_history["best_val_acc"], 4),
            "Test acc": round(rnn_test_acc, 4),
        },
        {
            "Model": "Transformer",
            "Parameters": count_parameters(transformer),
            "Train time (s)": round(transformer_history["total_time"], 1),
            "Best val acc": round(transformer_history["best_val_acc"], 4),
            "Test acc": round(transformer_test_acc, 4),
        },
        {
            "Model": "CNN (augmented)",
            "Parameters": count_parameters(cnn_aug),
            "Train time (s)": round(cnn_aug_history["total_time"], 1),
            "Best val acc": round(cnn_aug_history["best_val_acc"], 4),
            "Test acc": round(cnn_aug_clean_acc, 4),
        },
    ]
)
comparison


## 11. Discussion and honest conclusions

### 11.1 What worked

- All three architectures learn the 10-class problem well above the 10% chance
  floor. The CNN on rasters is the strongest in a full-drawing setting, which
  matches the intuition that the finished pixel grid carries the most signal
  for classification.
- The accuracy-vs-strokes curves show the expected monotonic climb: the
  Transformer and LSTM actually **have an edge in the very-early regime**
  (small `stroke_frac`), because a single stroke still lets them reason about
  direction and pen-lift structure, while the CNN only sees a mostly empty
  bitmap.
- Augmentation closes the robustness gap on the noisy evaluation. Without it,
  the CNN drops noticeably when rotation + noise + erasing are applied at test
  time. With augmentation, the drop is substantially smaller — the main point
  of the robustness experiment.

### 11.2 What to be careful about

- Some confusions are class-pair specific (e.g. *bicycle* vs *car*). This is a
  genuine ambiguity in the data, not a model failure.
- The Quick, Draw labels are the *intended* class, not the *rendered* class;
  `recognized=True` only tells us Google's original classifier agreed. This
  places an irreducible noise floor on accuracy.
- Results are reported on one seed. For a rigorous paper the experiment would
  be repeated over several seeds. Running three seeds would triple runtime,
  which did not fit within the course constraints.

### 11.3 Limitations

- The bitmap is tiny (28x28). Upscaling would help the CNN but also dwarfs the
  stroke models' complexity.
- Sequence length is capped at 128 points; very long drawings get truncated.
  This affects the tail of the accuracy-vs-strokes curve more than the start.
- `stroke_frac` is a coarse proxy for real-time progress: a single long stroke
  and ten short strokes both count as "one stroke". A finer protocol would
  slice by *time* rather than *stroke count*.

### 11.4 Takeaway

For the fully-drawn setting the CNN wins on raw accuracy and is the smallest
and fastest model. For the **real-time** regime the stroke-based models are
competitive and sometimes better early in the drawing, suggesting an
ensemble or a two-stage system (stroke model for early predictions, CNN once
enough of the drawing is visible) would be a natural follow-up. Augmentation
is a cheap, reliable robustness win.


## 12. Acknowledgements and code provenance

- Dataset: **Google Creative Lab — Quick, Draw!**
  <https://github.com/googlecreativelab/quickdraw-dataset>.
- PyTorch model primitives (`nn.Conv2d`, `nn.LSTM`, `nn.TransformerEncoder`,
  `nn.TransformerEncoderLayer`, `F.affine_grid`, `F.grid_sample`) are used as
  provided by the PyTorch library.
- The positional-encoding implementation follows the reference from Vaswani
  *et al.* 2017, "Attention Is All You Need".
- All model architectures (`SketchCNN`, `SketchRNN`, `SketchTransformer`),
  the data preprocessing (`strokes_to_image`, `strokes_to_sequence`), the
  augmentation class, and the training / evaluation utilities were written
  from scratch for this project.
- Code style was checked with `black` and `flake8`.
